# 01 — Data Preprocessing

## Scientific Abstract GPT

This notebook prepares the arXiv scientific-paper dataset for training a
decoder-only GPT-style Transformer from scratch.

### Processing steps

1. Load the arXiv dataset from Hugging Face.
2. Filter papers from selected computer-science domains.
3. Clean titles, subjects, and abstracts.
4. Add structural tokens:
   - `<TITLE>`
   - `<SUBJECT>`
   - `<ABSTRACT>`
   - `<END>`
5. Remove empty, very short, and duplicate records.
6. Create training, validation, and test splits.
7. Save the processed Hugging Face dataset.
8. Export plain-text files for BPE tokenizer training.


Install Required Libraries

In [1]:
!pip install -q datasets

Mount Google Drive

In [4]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


Import Libraries

In [5]:
import os
import re
import json
import random
import unicodedata

import pandas as pd

from datasets import (
    load_dataset,
    Dataset,
    DatasetDict
)

Set Random Seed

In [6]:
SEED = 42

random.seed(SEED)

print("Random seed:", SEED)

Random seed: 42


Define Project Paths

In [7]:
PROJECT_PATH = (
    "/content/drive/MyDrive/"
    "Scientific-Abstract-GPT"
)

DATA_FOLDER = os.path.join(
    PROJECT_PATH,
    "data"
)

PROCESSED_DATASET_PATH = os.path.join(
    DATA_FOLDER,
    "processed_structured_dataset"
)

TOKENIZER_DATA_PATH = os.path.join(
    DATA_FOLDER,
    "tokenizer_data"
)

TRAIN_TEXT_PATH = os.path.join(
    TOKENIZER_DATA_PATH,
    "train_text.txt"
)

VALIDATION_TEXT_PATH = os.path.join(
    TOKENIZER_DATA_PATH,
    "validation_text.txt"
)

TEST_TEXT_PATH = os.path.join(
    TOKENIZER_DATA_PATH,
    "test_text.txt"
)

PREPROCESSING_SUMMARY_PATH = os.path.join(
    DATA_FOLDER,
    "preprocessing_summary.json"
)

os.makedirs(
    DATA_FOLDER,
    exist_ok=True
)

os.makedirs(
    TOKENIZER_DATA_PATH,
    exist_ok=True
)

print("Project path:", PROJECT_PATH)
print("Processed dataset path:", PROCESSED_DATASET_PATH)
print("Tokenizer data path:", TOKENIZER_DATA_PATH)

Project path: /content/drive/MyDrive/Scientific-Abstract-GPT
Processed dataset path: /content/drive/MyDrive/Scientific-Abstract-GPT/data/processed_structured_dataset
Tokenizer data path: /content/drive/MyDrive/Scientific-Abstract-GPT/data/tokenizer_data


Define Dataset Configuration

In [8]:
DATASET_NAME = "nick007x/arxiv-papers"

SELECTED_CATEGORIES = {
    "Artificial Intelligence (cs.AI)",
    "Machine Learning (cs.LG)",
    "Computation and Language (cs.CL)"
}

SPECIAL_TOKENS = [
    "<PAD>",
    "<UNK>",
    "<TITLE>",
    "<SUBJECT>",
    "<ABSTRACT>",
    "<END>"
]

print("Dataset:", DATASET_NAME)

print("\nSelected categories:")

for category in sorted(SELECTED_CATEGORIES):
    print("-", category)

print("\nSpecial tokens:")

for token in SPECIAL_TOKENS:
    print("-", token)

Dataset: nick007x/arxiv-papers

Selected categories:
- Artificial Intelligence (cs.AI)
- Computation and Language (cs.CL)
- Machine Learning (cs.LG)

Special tokens:
- <PAD>
- <UNK>
- <TITLE>
- <SUBJECT>
- <ABSTRACT>
- <END>


Load the arXiv Dataset

In [9]:
raw_dataset = load_dataset(
    DATASET_NAME
)

print(raw_dataset)

train.parquet: reconstructing file:   0%|          |  0.00B / 1.70GB            

train.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2549619 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['arxiv_id', 'title', 'authors', 'submission_date', 'comments', 'primary_subject', 'subjects', 'doi', 'abstract', 'file_path'],
        num_rows: 2549619
    })
})


Select the Available Dataset Split

In [10]:
print("Available splits:", list(raw_dataset.keys()))

if "train" not in raw_dataset:
    raise KeyError(
        "The dataset does not contain a train split."
    )

dataset = raw_dataset["train"]

print("\nTotal records before filtering:")
print(f"{len(dataset):,}")

Available splits: ['train']

Total records before filtering:
2,549,619


Inspect Dataset Columns

In [11]:
print("Dataset columns:")

for column in dataset.column_names:
    print("-", column)

Dataset columns:
- arxiv_id
- title
- authors
- submission_date
- comments
- primary_subject
- subjects
- doi
- abstract
- file_path


Display One Original Record

In [12]:
sample_record = dataset[0]

print("Title:")
print(sample_record.get("title"))

print("\nPrimary subject:")
print(sample_record.get("primary_subject"))

print("\nAbstract:")
print(sample_record.get("abstract"))

Title:
The gravitational wave background from star-massive black hole fly-bys

Primary subject:
Earth and Planetary Astrophysics (astro-ph.EP)

Abstract:
Stars on eccentric orbits around a massive black hole (MBH) emit bursts of gravitational waves (GWs) at periapse. Such events may be directly resolvable in the Galactic centre. However, if the star does not spiral in, the emitted GWs are not resolvable for extra-galactic MBHs, but constitute a source of background noise. We estimate the power spectrum of this extreme mass ratio burst background (EMBB) and compare it to the anticipated instrumental noise of the Laser Interferometer Space Antenna (LISA). To this end, we model the regions close to a MBH, accounting for mass-segregation, and for processes that limit the presence of stars close to the MBH, such as GW inspiral and hydrodynamical collisions between stars. We find that the EMBB is dominated by GW bursts from stellar mass black holes, and the magnitude of the noise spectrum (f

Verify Required Columns

In [13]:
REQUIRED_COLUMNS = {
    "title",
    "abstract",
    "primary_subject"
}

missing_columns = (
    REQUIRED_COLUMNS -
    set(dataset.column_names)
)

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        f"{sorted(missing_columns)}"
    )

print("All required columns are available.")

All required columns are available.


Count Papers by Selected Category

In [14]:
category_counts_before_filtering = {}

for category in sorted(SELECTED_CATEGORIES):

    count = sum(
        1
        for subject in dataset["primary_subject"]
        if subject == category
    )

    category_counts_before_filtering[
        category
    ] = count

    print(
        f"{category}: {count:,}"
    )

Artificial Intelligence (cs.AI): 27,726
Computation and Language (cs.CL): 64,648
Machine Learning (cs.LG): 106,484


Filter Selected Categories

In [15]:
def keep_selected_category(example):
    """
    Keep only AI, Machine Learning, and
    Computation and Language papers.
    """

    return (
        example.get("primary_subject")
        in SELECTED_CATEGORIES
    )

In [16]:
filtered_dataset = dataset.filter(
    keep_selected_category,
    desc="Filtering selected categories"
)

print("Records before filtering:")
print(f"{len(dataset):,}")

print("\nRecords after category filtering:")
print(f"{len(filtered_dataset):,}")

Filtering selected categories:   0%|          | 0/2549619 [00:00<?, ? examples/s]

Records before filtering:
2,549,619

Records after category filtering:
198,858


Define Text-Cleaning Function

In [17]:
def clean_text(value):
    """
    Clean a title, subject, or abstract.

    Processing:
    - Convert missing values to an empty string
    - Normalize Unicode
    - Remove HTML-like tags
    - Replace line breaks and tabs with spaces
    - Remove structural-token characters from source text
    - Collapse repeated whitespace
    """

    if value is None:
        return ""

    text = str(value)

    text = unicodedata.normalize(
        "NFKC",
        text
    )

    text = re.sub(
        r"<[^>]+>",
        " ",
        text
    )

    text = text.replace(
        "\n",
        " "
    )

    text = text.replace(
        "\r",
        " "
    )

    text = text.replace(
        "\t",
        " "
    )

    # Avoid accidental structural tokens
    # appearing inside the original text.
    text = text.replace(
        "<",
        " "
    )

    text = text.replace(
        ">",
        " "
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

Test Text Cleaning

In [18]:
unclean_text = (
    "  Deep   Learning\n"
    "for\tMedical <b>Imaging</b>  "
)

cleaned_text = clean_text(
    unclean_text
)

print("Before:")
print(repr(unclean_text))

print("\nAfter:")
print(repr(cleaned_text))

Before:
'  Deep   Learning\nfor\tMedical <b>Imaging</b>  '

After:
'Deep Learning for Medical Imaging'


Create Simplified Subject Labels

In [19]:
SUBJECT_LABEL_MAP = {
    "Artificial Intelligence (cs.AI)": (
        "Artificial Intelligence"
    ),
    "Machine Learning (cs.LG)": (
        "Machine Learning"
    ),
    "Computation and Language (cs.CL)": (
        "Computation and Language"
    )
}

print(SUBJECT_LABEL_MAP)

{'Artificial Intelligence (cs.AI)': 'Artificial Intelligence', 'Machine Learning (cs.LG)': 'Machine Learning', 'Computation and Language (cs.CL)': 'Computation and Language'}


Define Structured Preprocessing Function

In [20]:
def preprocess_record(example):
    """
    Clean one scientific-paper record and
    create a structured training sequence.
    """

    title = clean_text(
        example.get("title", "")
    )

    abstract = clean_text(
        example.get("abstract", "")
    )

    original_subject = clean_text(
        example.get("primary_subject", "")
    )

    subject = SUBJECT_LABEL_MAP.get(
        original_subject,
        original_subject
    )

    structured_text = (
        f"<TITLE> {title} "
        f"<SUBJECT> {subject} "
        f"<ABSTRACT> {abstract} "
        f"<END>"
    )

    return {
        "clean_title": title,
        "clean_subject": subject,
        "clean_abstract": abstract,
        "text": structured_text,
        "title_length": len(title),
        "abstract_length": len(abstract),
        "abstract_word_count": len(
            abstract.split()
        )
    }

Apply Preprocessing

In [21]:
processed_dataset = filtered_dataset.map(
    preprocess_record,
    desc="Cleaning and structuring records"
)

print(processed_dataset)

Cleaning and structuring records:   0%|          | 0/198858 [00:00<?, ? examples/s]

Dataset({
    features: ['arxiv_id', 'title', 'authors', 'submission_date', 'comments', 'primary_subject', 'subjects', 'doi', 'abstract', 'file_path', 'clean_title', 'clean_subject', 'clean_abstract', 'text', 'title_length', 'abstract_length', 'abstract_word_count'],
    num_rows: 198858
})


Inspect a Structured Record

In [22]:
example = processed_dataset[0]

print("Clean title:")
print(example["clean_title"])

print("\nClean subject:")
print(example["clean_subject"])

print("\nClean abstract:")
print(example["clean_abstract"][:500])

print("\nStructured text:")
print(example["text"][:1000])

Clean title:
Parametric Learning and Monte Carlo Optimization

Clean subject:
Machine Learning

Clean abstract:
This paper uncovers and explores the close relationship between Monte Carlo Optimization of a parametrized integral (MCO), Parametric machine-Learning (PL), and `blackbox&#39; or `oracle&#39;-based optimization (BO). We make four contributions. First, we prove that MCO is mathematically identical to a broad class of PL problems. This identity potentially provides a new application domain for all broadly applicable PL techniques: MCO. Second, we introduce immediate sampling, a new version of the 

Structured text:
<TITLE> Parametric Learning and Monte Carlo Optimization <SUBJECT> Machine Learning <ABSTRACT> This paper uncovers and explores the close relationship between Monte Carlo Optimization of a parametrized integral (MCO), Parametric machine-Learning (PL), and `blackbox&#39; or `oracle&#39;-based optimization (BO). We make four contributions. First, we prove that MCO is m

Define Quality Filter

In [23]:
MIN_TITLE_CHARACTERS = 5
MIN_ABSTRACT_WORDS = 40
MAX_ABSTRACT_WORDS = 600

print(
    "Minimum title characters:",
    MIN_TITLE_CHARACTERS
)

print(
    "Minimum abstract words:",
    MIN_ABSTRACT_WORDS
)

print(
    "Maximum abstract words:",
    MAX_ABSTRACT_WORDS
)

Minimum title characters: 5
Minimum abstract words: 40
Maximum abstract words: 600


In [24]:
def is_valid_record(example):
    """
    Keep records with usable titles and abstracts.
    """

    title = example["clean_title"]
    abstract = example["clean_abstract"]
    word_count = example["abstract_word_count"]

    if not title:
        return False

    if not abstract:
        return False

    if len(title) < MIN_TITLE_CHARACTERS:
        return False

    if word_count < MIN_ABSTRACT_WORDS:
        return False

    if word_count > MAX_ABSTRACT_WORDS:
        return False

    return True

Remove Invalid Records

In [25]:
records_before_quality_filter = len(
    processed_dataset
)

processed_dataset = processed_dataset.filter(
    is_valid_record,
    desc="Removing invalid records"
)

records_after_quality_filter = len(
    processed_dataset
)

print("Before quality filtering:")
print(f"{records_before_quality_filter:,}")

print("\nAfter quality filtering:")
print(f"{records_after_quality_filter:,}")

print("\nRemoved records:")
print(
    f"{records_before_quality_filter - records_after_quality_filter:,}"
)

Removing invalid records:   0%|          | 0/198858 [00:00<?, ? examples/s]

Before quality filtering:
198,858

After quality filtering:
198,406

Removed records:
452


Keep Only Required Columns

In [26]:
COLUMNS_TO_KEEP = [
    "clean_title",
    "clean_subject",
    "clean_abstract",
    "text",
    "title_length",
    "abstract_length",
    "abstract_word_count"
]

columns_to_remove = [
    column
    for column in processed_dataset.column_names
    if column not in COLUMNS_TO_KEEP
]

processed_dataset = (
    processed_dataset.remove_columns(
        columns_to_remove
    )
)

print("Final columns:")

for column in processed_dataset.column_names:
    print("-", column)

Final columns:
- clean_title
- clean_subject
- clean_abstract
- text
- title_length
- abstract_length
- abstract_word_count


Remove Duplicate Records

In [27]:
processed_dataframe = (
    processed_dataset.to_pandas()
)

print(
    "Records before duplicate removal:",
    f"{len(processed_dataframe):,}"
)

processed_dataframe = (
    processed_dataframe.drop_duplicates(
        subset=[
            "clean_title",
            "clean_abstract"
        ]
    )
)

processed_dataframe = (
    processed_dataframe.reset_index(
        drop=True
    )
)

print(
    "Records after duplicate removal:",
    f"{len(processed_dataframe):,}"
)

Records before duplicate removal: 198,406
Records after duplicate removal: 198,388


Convert Back to Hugging Face Dataset

In [28]:
processed_dataset = Dataset.from_pandas(
    processed_dataframe,
    preserve_index=False
)

print(processed_dataset)

Dataset({
    features: ['clean_title', 'clean_subject', 'clean_abstract', 'text', 'title_length', 'abstract_length', 'abstract_word_count'],
    num_rows: 198388
})


Shuffle the Dataset

In [29]:
processed_dataset = (
    processed_dataset.shuffle(
        seed=SEED
    )
)

print(
    "Dataset shuffled with seed:",
    SEED
)

Dataset shuffled with seed: 42


Show Dataset Statistics

In [30]:
statistics = {
    "total_records": len(
        processed_dataset
    ),
    "average_title_characters": (
        sum(
            processed_dataset[
                "title_length"
            ]
        ) /
        len(processed_dataset)
    ),
    "average_abstract_characters": (
        sum(
            processed_dataset[
                "abstract_length"
            ]
        ) /
        len(processed_dataset)
    ),
    "average_abstract_words": (
        sum(
            processed_dataset[
                "abstract_word_count"
            ]
        ) /
        len(processed_dataset)
    )
}

print(
    "Total usable records:",
    f"{statistics['total_records']:,}"
)

print(
    "Average title characters:",
    f"{statistics['average_title_characters']:.2f}"
)

print(
    "Average abstract characters:",
    f"{statistics['average_abstract_characters']:.2f}"
)

print(
    "Average abstract words:",
    f"{statistics['average_abstract_words']:.2f}"
)

Total usable records: 198,388
Average title characters: 75.33
Average abstract characters: 1168.66
Average abstract words: 165.79


Display Subject Distribution

In [31]:
subject_distribution = (
    pd.Series(
        processed_dataset[
            "clean_subject"
        ]
    )
    .value_counts()
    .rename_axis("Subject")
    .reset_index(name="Count")
)

subject_distribution[
    "Percentage"
] = (
    subject_distribution["Count"] /
    subject_distribution["Count"].sum()
    * 100
)

subject_distribution

,Subject,Count,Percentage
0,Machine Learning,106311,53.587415
1,Computation and Language,64534,32.529185
2,Artificial Intelligence,27543,13.883400


Create Train, Validation, and Test Splits

In [32]:
train_and_remaining = (
    processed_dataset.train_test_split(
        test_size=0.10,
        seed=SEED
    )
)

remaining_split = (
    train_and_remaining[
        "test"
    ].train_test_split(
        test_size=0.50,
        seed=SEED
    )
)

dataset_splits = DatasetDict({
    "train": train_and_remaining[
        "train"
    ],
    "validation": remaining_split[
        "train"
    ],
    "test": remaining_split[
        "test"
    ]
})

print(dataset_splits)

DatasetDict({
    train: Dataset({
        features: ['clean_title', 'clean_subject', 'clean_abstract', 'text', 'title_length', 'abstract_length', 'abstract_word_count'],
        num_rows: 178549
    })
    validation: Dataset({
        features: ['clean_title', 'clean_subject', 'clean_abstract', 'text', 'title_length', 'abstract_length', 'abstract_word_count'],
        num_rows: 9919
    })
    test: Dataset({
        features: ['clean_title', 'clean_subject', 'clean_abstract', 'text', 'title_length', 'abstract_length', 'abstract_word_count'],
        num_rows: 9920
    })
})


Verify Split Sizes

In [33]:
total_records = sum(
    len(split)
    for split in dataset_splits.values()
)

print(
    "Training records:",
    f"{len(dataset_splits['train']):,}"
)

print(
    "Validation records:",
    f"{len(dataset_splits['validation']):,}"
)

print(
    "Testing records:",
    f"{len(dataset_splits['test']):,}"
)

print(
    "\nTotal records:",
    f"{total_records:,}"
)

Training records: 178,549
Validation records: 9,919
Testing records: 9,920

Total records: 198,388


Calculate Split Percentages

In [34]:
for split_name, split_data in (
    dataset_splits.items()
):

    percentage = (
        len(split_data) /
        total_records *
        100
    )

    print(
        f"{split_name.capitalize()}: "
        f"{len(split_data):,} "
        f"({percentage:.2f}%)"
    )

Train: 178,549 (90.00%)
Validation: 9,919 (5.00%)
Test: 9,920 (5.00%)


Inspect One Example from Each Split

In [35]:
for split_name in [
    "train",
    "validation",
    "test"
]:

    print("=" * 100)
    print(split_name.upper())
    print("=" * 100)

    print(
        dataset_splits[
            split_name
        ][0]["text"][:1000]
    )

    print()

TRAIN
<TITLE> Contrastive Language Adaptation for Cross-Lingual Stance Detection <SUBJECT> Computation and Language <ABSTRACT> We study cross-lingual stance detection, which aims to leverage labeled data in one language to identify the relative perspective (or stance) of a given document with respect to a claim in a different target language. In particular, we introduce a novel contrastive language adaptation approach applied to memory networks, which ensures accurate alignment of stances in the source and target languages, and can effectively deal with the challenge of limited labeled data in the target language. The evaluation results on public benchmark datasets and comparison against current state-of-the-art approaches demonstrate the effectiveness of our approach. <END>

VALIDATION
<TITLE> WebLINX: Real-World Website Navigation with Multi-Turn Dialogue <SUBJECT> Computation and Language <ABSTRACT> We propose the problem of conversational web navigation, where a digital agent contr

Verify Structural Tokens

In [36]:
def verify_structure(text):
    """
    Verify that all required structural markers
    exist and appear in the correct order.
    """

    required_tokens = [
        "<TITLE>",
        "<SUBJECT>",
        "<ABSTRACT>",
        "<END>"
    ]

    if not all(
        token in text
        for token in required_tokens
    ):
        return False

    positions = [
        text.find(token)
        for token in required_tokens
    ]

    return positions == sorted(
        positions
    )

In [37]:
sample_size = min(
    1000,
    len(dataset_splits["train"])
)

valid_structure_count = sum(
    verify_structure(
        dataset_splits[
            "train"
        ][index]["text"]
    )
    for index in range(sample_size)
)

print(
    "Records checked:",
    sample_size
)

print(
    "Records with valid structure:",
    valid_structure_count
)

print(
    "Structural compliance:",
    f"{valid_structure_count / sample_size * 100:.2f}%"
)

Records checked: 1000
Records with valid structure: 1000
Structural compliance: 100.00%


Remove Existing Saved Dataset Safely

In [38]:
import shutil

if os.path.exists(
    PROCESSED_DATASET_PATH
):

    print(
        "Removing existing processed dataset:"
    )

    print(
        PROCESSED_DATASET_PATH
    )

    shutil.rmtree(
        PROCESSED_DATASET_PATH
    )

Save Processed Hugging Face Dataset

In [39]:
dataset_splits.save_to_disk(
    PROCESSED_DATASET_PATH
)

print(
    "Processed dataset saved successfully."
)

print(
    "Location:",
    PROCESSED_DATASET_PATH
)

Saving the dataset (0/1 shards):   0%|          | 0/178549 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9919 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9920 [00:00<?, ? examples/s]

Processed dataset saved successfully.
Location: /content/drive/MyDrive/Scientific-Abstract-GPT/data/processed_structured_dataset


Define Text Export Function

In [40]:
def export_text_file(
    dataset_split,
    output_path
):
    """
    Export the structured-text column to
    a UTF-8 plain-text file.
    """

    record_count = 0

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as output_file:

        for example in dataset_split:

            text = example[
                "text"
            ].strip()

            if not text:
                continue

            output_file.write(
                text + "\n"
            )

            record_count += 1

    return record_count

Export Training Text

In [41]:
train_export_count = export_text_file(
    dataset_splits["train"],
    TRAIN_TEXT_PATH
)

print(
    "Training records exported:",
    f"{train_export_count:,}"
)

print(
    "Training text path:",
    TRAIN_TEXT_PATH
)

Training records exported: 178,549
Training text path: /content/drive/MyDrive/Scientific-Abstract-GPT/data/tokenizer_data/train_text.txt


Export Validation Text

In [42]:
validation_export_count = (
    export_text_file(
        dataset_splits[
            "validation"
        ],
        VALIDATION_TEXT_PATH
    )
)

print(
    "Validation records exported:",
    f"{validation_export_count:,}"
)

print(
    "Validation text path:",
    VALIDATION_TEXT_PATH
)

Validation records exported: 9,919
Validation text path: /content/drive/MyDrive/Scientific-Abstract-GPT/data/tokenizer_data/validation_text.txt


Export Test Text

In [43]:
test_export_count = export_text_file(
    dataset_splits["test"],
    TEST_TEXT_PATH
)

print(
    "Test records exported:",
    f"{test_export_count:,}"
)

print(
    "Test text path:",
    TEST_TEXT_PATH
)

Test records exported: 9,920
Test text path: /content/drive/MyDrive/Scientific-Abstract-GPT/data/tokenizer_data/test_text.txt


Verify Exported Files

In [44]:
exported_files = {
    "train": TRAIN_TEXT_PATH,
    "validation": VALIDATION_TEXT_PATH,
    "test": TEST_TEXT_PATH
}

for split_name, file_path in (
    exported_files.items()
):

    exists = os.path.exists(
        file_path
    )

    file_size_mb = (
        os.path.getsize(file_path) /
        (1024 ** 2)
        if exists
        else 0
    )

    print(
        f"{split_name.capitalize()} file"
    )

    print("Exists:", exists)

    print(
        f"Size: {file_size_mb:.2f} MB"
    )

    print("-" * 50)

Train file
Exists: True
Size: 221.68 MB
--------------------------------------------------
Validation file
Exists: True
Size: 12.29 MB
--------------------------------------------------
Test file
Exists: True
Size: 12.28 MB
--------------------------------------------------


Preview Exported Training Text

In [45]:
with open(
    TRAIN_TEXT_PATH,
    "r",
    encoding="utf-8"
) as file:

    for index in range(3):

        line = file.readline()

        print(
            f"Training example {index + 1}:"
        )

        print(
            line[:1000]
        )

        print("=" * 100)

Training example 1:
<TITLE> Contrastive Language Adaptation for Cross-Lingual Stance Detection <SUBJECT> Computation and Language <ABSTRACT> We study cross-lingual stance detection, which aims to leverage labeled data in one language to identify the relative perspective (or stance) of a given document with respect to a claim in a different target language. In particular, we introduce a novel contrastive language adaptation approach applied to memory networks, which ensures accurate alignment of stances in the source and target languages, and can effectively deal with the challenge of limited labeled data in the target language. The evaluation results on public benchmark datasets and comparison against current state-of-the-art approaches demonstrate the effectiveness of our approach. <END>

Training example 2:
<TITLE> Do Neural Networks Need Gradient Descent to Generalize? A Theoretical Study <SUBJECT> Machine Learning <ABSTRACT> Conventional wisdom attributes the mysterious generalizat

Check for Empty Exported Lines

In [46]:
def count_empty_lines(file_path):

    empty_line_count = 0
    total_line_count = 0

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as file:

        for line in file:

            total_line_count += 1

            if not line.strip():
                empty_line_count += 1

    return (
        total_line_count,
        empty_line_count
    )

Save Preprocessing Summary

In [47]:
for split_name, file_path in (
    exported_files.items()
):

    total_lines, empty_lines = (
        count_empty_lines(
            file_path
        )
    )

    print(
        f"{split_name.capitalize()}: "
        f"{total_lines:,} total lines, "
        f"{empty_lines:,} empty lines"
    )

Train: 178,549 total lines, 0 empty lines
Validation: 9,919 total lines, 0 empty lines
Test: 9,920 total lines, 0 empty lines


In [48]:
subject_counts = {
    row["Subject"]: int(
        row["Count"]
    )
    for _, row in (
        subject_distribution.iterrows()
    )
}

preprocessing_summary = {
    "dataset_name": DATASET_NAME,
    "seed": SEED,
    "selected_categories": sorted(
        SELECTED_CATEGORIES
    ),
    "subject_label_map": (
        SUBJECT_LABEL_MAP
    ),
    "special_tokens": (
        SPECIAL_TOKENS
    ),
    "quality_filters": {
        "minimum_title_characters": (
            MIN_TITLE_CHARACTERS
        ),
        "minimum_abstract_words": (
            MIN_ABSTRACT_WORDS
        ),
        "maximum_abstract_words": (
            MAX_ABSTRACT_WORDS
        )
    },
    "records": {
        "after_category_filter": (
            len(filtered_dataset)
        ),
        "after_quality_and_duplicate_filter": (
            len(processed_dataset)
        ),
        "train": len(
            dataset_splits["train"]
        ),
        "validation": len(
            dataset_splits[
                "validation"
            ]
        ),
        "test": len(
            dataset_splits["test"]
        )
    },
    "statistics": {
        "average_title_characters": round(
            statistics[
                "average_title_characters"
            ],
            2
        ),
        "average_abstract_characters": round(
            statistics[
                "average_abstract_characters"
            ],
            2
        ),
        "average_abstract_words": round(
            statistics[
                "average_abstract_words"
            ],
            2
        )
    },
    "subject_distribution": (
        subject_counts
    ),
    "output_paths": {
        "processed_dataset": (
            PROCESSED_DATASET_PATH
        ),
        "train_text": (
            TRAIN_TEXT_PATH
        ),
        "validation_text": (
            VALIDATION_TEXT_PATH
        ),
        "test_text": (
            TEST_TEXT_PATH
        )
    }
}

with open(
    PREPROCESSING_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        preprocessing_summary,
        file,
        indent=2,
        ensure_ascii=False
    )

print(
    "Preprocessing summary saved:"
)

print(
    PREPROCESSING_SUMMARY_PATH
)

Preprocessing summary saved:
/content/drive/MyDrive/Scientific-Abstract-GPT/data/preprocessing_summary.json


Display Preprocessing Summary

In [49]:
print(
    json.dumps(
        preprocessing_summary,
        indent=2,
        ensure_ascii=False
    )
)

{
  "dataset_name": "nick007x/arxiv-papers",
  "seed": 42,
  "selected_categories": [
    "Artificial Intelligence (cs.AI)",
    "Computation and Language (cs.CL)",
    "Machine Learning (cs.LG)"
  ],
  "subject_label_map": {
    "Artificial Intelligence (cs.AI)": "Artificial Intelligence",
    "Machine Learning (cs.LG)": "Machine Learning",
    "Computation and Language (cs.CL)": "Computation and Language"
  },
  "special_tokens": [
    "<PAD>",
    "<UNK>",
    "<TITLE>",
    "<SUBJECT>",
    "<ABSTRACT>",
    "<END>"
  ],
  "quality_filters": {
    "minimum_title_characters": 5,
    "minimum_abstract_words": 40,
    "maximum_abstract_words": 600
  },
  "records": {
    "after_category_filter": 198858,
    "after_quality_and_duplicate_filter": 198388,
    "train": 178549,
    "validation": 9919,
    "test": 9920
  },
  "statistics": {
    "average_title_characters": 75.33,
    "average_abstract_characters": 1168.66,
    "average_abstract_words": 165.79
  },
  "subject_distribution": 

Display Preprocessing Summary

In [50]:
from datasets import load_from_disk

reloaded_dataset = load_from_disk(
    PROCESSED_DATASET_PATH
)

print(reloaded_dataset)

print("\nReloaded training example:")

print(
    reloaded_dataset[
        "train"
    ][0]["text"][:1000]
)

DatasetDict({
    train: Dataset({
        features: ['clean_title', 'clean_subject', 'clean_abstract', 'text', 'title_length', 'abstract_length', 'abstract_word_count'],
        num_rows: 178549
    })
    validation: Dataset({
        features: ['clean_title', 'clean_subject', 'clean_abstract', 'text', 'title_length', 'abstract_length', 'abstract_word_count'],
        num_rows: 9919
    })
    test: Dataset({
        features: ['clean_title', 'clean_subject', 'clean_abstract', 'text', 'title_length', 'abstract_length', 'abstract_word_count'],
        num_rows: 9920
    })
})

Reloaded training example:
<TITLE> Contrastive Language Adaptation for Cross-Lingual Stance Detection <SUBJECT> Computation and Language <ABSTRACT> We study cross-lingual stance detection, which aims to leverage labeled data in one language to identify the relative perspective (or stance) of a given document with respect to a claim in a different target language. In particular, we introduce a novel contrastive l

Final Validation

In [51]:
validation_checks = {
    "processed_dataset_saved": (
        os.path.exists(
            PROCESSED_DATASET_PATH
        )
    ),
    "train_text_saved": (
        os.path.exists(
            TRAIN_TEXT_PATH
        )
    ),
    "validation_text_saved": (
        os.path.exists(
            VALIDATION_TEXT_PATH
        )
    ),
    "test_text_saved": (
        os.path.exists(
            TEST_TEXT_PATH
        )
    ),
    "summary_saved": (
        os.path.exists(
            PREPROCESSING_SUMMARY_PATH
        )
    ),
    "train_split_not_empty": (
        len(dataset_splits["train"]) > 0
    ),
    "validation_split_not_empty": (
        len(
            dataset_splits[
                "validation"
            ]
        ) > 0
    ),
    "test_split_not_empty": (
        len(dataset_splits["test"]) > 0
    ),
    "structure_valid": (
        valid_structure_count
        == sample_size
    )
}

for check_name, result in (
    validation_checks.items()
):

    status = (
        "PASSED"
        if result
        else "FAILED"
    )

    print(
        f"{check_name}: {status}"
    )

processed_dataset_saved: PASSED
train_text_saved: PASSED
validation_text_saved: PASSED
test_text_saved: PASSED
summary_saved: PASSED
train_split_not_empty: PASSED
validation_split_not_empty: PASSED
test_split_not_empty: PASSED
structure_valid: PASSED


Completion Message

In [53]:
if all(validation_checks.values()):

    print("=" * 80)

    print(
        "01_Data_Preprocessing.ipynb "
        "completed successfully."
    )

    print("=" * 80)

    print(
        "\nThe structured dataset contains:"
    )

    print(
        "<TITLE>, <SUBJECT>, "
        "<ABSTRACT>, and <END> tokens."
    )

else:

    failed_checks = [
        check_name
        for check_name, result
        in validation_checks.items()
        if not result
    ]

    raise RuntimeError(
        "Preprocessing validation failed: "
        f"{failed_checks}"
    )

01_Data_Preprocessing.ipynb completed successfully.

The structured dataset contains:
<TITLE>, <SUBJECT>, <ABSTRACT>, and <END> tokens.
